# Tema: Joins, agregaciones y diagnóstico

## Objetivos
Elegir joins, calcular métricas y leer planes para detectar shuffles.

## Conceptos importantes para el examen
Inner/left/anti/cross; join multiclave; broadcast; skew y spill; count, approx_count_distinct y mean.

**Dificultad:** Intermedio · **Tiempo estimado:** 70 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_12_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("errorifexists").saveAsTable("employees")
display(employees.orderBy("employee_id"))

In [ ]:
departments = spark.createDataFrame([("Data", "Tecnología"), ("Sales", "Comercial"), ("Legal", "Corporativo")], "department STRING, area STRING")
departments.createOrReplaceTempView("departments")

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Inner y left

In [ ]:
display(employees.join(departments, "department", "inner"))
display(employees.join(departments, "department", "left"))

### 2. Broadcast de una dimensión pequeña

In [ ]:
joined = employees.join(F.broadcast(departments), "department", "left")
joined.explain("formatted")
print(joined.count())

### 3. Estadísticas

In [ ]:
display(employees.groupBy("department").agg(F.count("*").alias("n"), F.mean("salary").alias("mean_salary"), F.approx_count_distinct("employee_id").alias("approx_people")))
display(employees.select("salary").summary())

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Localiza empleados cuyo departamento no existe en departments.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Calcula masa salarial por area conservando empleados sin correspondencia.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Construye todas las combinaciones entre tres departamentos y dos años; verifica 6 filas.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Crea targets(department,year,target) y únelos por ambas claves con las combinaciones anteriores.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Compara el plan normal y con broadcast, registra la configuración disponible y abre métricas de ejecución. Explica skew, shuffle y spill sin atribuir velocidad a 18 filas.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** left_anti.

**Pista 2:** left y coalesce de etiquetas.

**Pista 3:** crossJoin solo es razonable aquí por el tamaño.

**Pista 4:** Lista de columnas en join.

**Pista 5:** AQE puede elegir broadcast por sí solo; parámetros no siempre editables en serverless.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
missing = employees.join(departments, "department", "left_anti")
assert missing.count() == 6
display(missing)

### Solución 2

In [ ]:
display(employees.join(departments, "department", "left").withColumn("area", F.coalesce("area", F.lit("Sin área"))).groupBy("area").agg(F.sum("salary").alias("payroll")))

### Solución 3

In [ ]:
years = spark.createDataFrame([(2025,), (2026,)], "year INT")
result = departments.crossJoin(years)
assert result.count() == 6
display(result)

### Solución 4

In [ ]:
targets = spark.createDataFrame([("Data",2026,100),("Sales",2026,200)], "department STRING, year INT, target INT")
display(result.join(targets, ["department", "year"], "left"))

### Solución 5

In [ ]:
employees.join(departments, "department").explain("formatted")
employees.join(F.broadcast(departments), "department").explain("formatted")
for key in ["spark.sql.shuffle.partitions", "spark.sql.autoBroadcastJoinThreshold", "spark.default.parallelism", "spark.executor.memory", "spark.driver.memory"]:
    try:
        print(key, spark.conf.get(key))
    except Exception:
        print(key, "no expuesto en este cómputo; consultar configuración/UI")
# En cómputo clásico: Spark UI → SQL/stages; compara tamaño de shuffle,
# distribución de duración entre tareas y bytes de spill.
# Skew: unas claves concentran trabajo. Spill: memoria insuficiente para la operación.
# Cambia un parámetro permitido de uno en uno y vuelve a medir con datos representativos.
# Serverless administra varias opciones; usa perfil de consulta y métricas disponibles.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué join localiza claves sin correspondencia?

A. cross

B. left_anti

C. inner

D. right siempre

### Pregunta 2
¿Qué candidato es adecuado para broadcast?

A. Tabla masiva de hechos

B. Todas las tablas siempre

C. Dimensión pequeña

D. Solo archivos JSON

### Pregunta 3
Una tarea dura mucho más que las demás y procesa muchos más datos. ¿Qué sospechas?

A. Skew de datos

B. Falta de un alias

C. Un comentario SQL

D. Uso de NULL siempre

### Respuestas y explicación
**1. B** — Devuelve filas izquierdas sin coincidencia.

**2. C** — La dimensión debe caber razonablemente en memoria.

**3. A** — La distribución desigual puede concentrar trabajo.

## PARTE 6 - RETO FINAL
Genera 10.000 pedidos con una clave muy frecuente, únelos a una dimensión pequeña y compara planes y métricas. Distingue evidencia de skew de una simple diferencia de tiempo.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
